In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("andrihjonior/cert-insider-threat-dataset-r4-2")

print("Path to dataset files:", path)

100%|██████████| 6.65G/6.65G [01:19<00:00, 89.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/versions/1


In [38]:
import pandas as pd
import numpy as np
import datetime
import os

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import classification_report



In [3]:
path = "/root/.cache/kagglehub/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/versions/1"
dfDevice = pd.read_csv(os.path.join(path, "r4.2", "device.csv"))
dfLogon = pd.read_csv(os.path.join(path, "r4.2", "logon.csv"))


In [4]:
dfDevice["date"] = pd.to_datetime(dfDevice["date"])
dfDevice["dateOnly"] = dfDevice["date"].dt.date
dfDevice["timeOnly"] = dfDevice["date"].dt.time
dfDevice.drop("date",axis=1,inplace=True)


dfLogon["date"] = pd.to_datetime(dfLogon["date"])
dfLogon["dateOnly"] = dfLogon["date"].dt.date
dfLogon["timeOnly"] = dfLogon["date"].dt.time
dfLogon.drop("date",axis=1,inplace=True)

In [5]:
dfDeviceAggregated = dfDevice.groupby(["user",'dateOnly']).agg(numDeviceEvents=("activity", "count"))
dfDeviceAggregated.reset_index(inplace=True)

In [6]:
start_time = datetime.time(8, 0)
end_time = datetime.time(18, 0)

# Apply the lambda function
dfLogon["isOffHours"] = dfLogon["timeOnly"].apply(
    lambda t: t < start_time or t > end_time
)
dfLogonAggregated = dfLogon.groupby(["user", "dateOnly"]).agg(
    numLogonEvents=("activity", "count"),
    numOffHourLogons=("isOffHours", "sum")
)
dfLogonAggregated.reset_index(inplace=True)

In [7]:
mergedAggregated = dfDeviceAggregated.merge(dfLogonAggregated,how="outer",on=["user","dateOnly"])
mergedAggregated.fillna(0,inplace=True)
mergedAggregated.head()

,user,dateOnly,numDeviceEvents,numLogonEvents,numOffHourLogons
0,AAE0190,2010-01-04,0.0,2,1
1,AAE0190,2010-01-05,0.0,2,1
2,AAE0190,2010-01-06,0.0,2,1
3,AAE0190,2010-01-07,0.0,2,1
4,AAE0190,2010-01-08,0.0,2,1


In [8]:
dfAnswers = pd.read_csv(os.path.join(path, "answers", "insiders.csv"))

In [9]:
dfAnswers[dfAnswers["dataset"]==4.2].head()

,dataset,scenario,details,user,start,end
8,4.2,1,r4.2-1-AAM0658.csv,AAM0658,10/23/2010 01:34:19,10/29/2010 05:23:28
9,4.2,1,r4.2-1-AJR0932.csv,AJR0932,09/10/2010 19:12:01,09/18/2010 02:02:51
10,4.2,1,r4.2-1-BDV0168.csv,BDV0168,07/30/2010 19:56:44,08/10/2010 05:16:41
11,4.2,1,r4.2-1-BIH0745.csv,BIH0745,07/13/2010 20:15:23,07/13/2010 21:20:44
12,4.2,1,r4.2-1-BLS0678.csv,BLS0678,09/21/2010 01:16:22,09/30/2010 04:48:19


In [10]:
userStats = mergedAggregated.groupby("user")[["numDeviceEvents", "numLogonEvents", "numOffHourLogons"]].agg(['mean', 'std'])
userStats.columns = [f"{col[0]}_{col[1]}" for col in userStats.columns]

# 2. Reset the index so 'user' is a regular column, not the index
userStats = userStats.reset_index()
userStats.head()

,user,numDeviceEvents_mean,numDeviceEvents_std,numLogonEvents_mean,numLogonEvents_std,numOffHourLogons_mean,numOffHourLogons_std
0,AAE0190,0.000000,0.000000,2.000000,0.000000,1.00,0.000000
1,AAF0535,4.195122,2.580907,2.000000,0.000000,0.00,0.000000
2,AAF0791,0.000000,0.000000,2.000000,0.000000,0.00,0.000000
3,AAL0706,0.000000,0.000000,2.000000,0.000000,1.00,0.000000
4,AAM0658,0.057778,0.464109,2.035556,0.264875,1.04,0.272554


In [11]:
dfZscore = mergedAggregated.merge(userStats,how="outer",on="user")
dfZscore.head()

,user,dateOnly,numDeviceEvents,numLogonEvents,numOffHourLogons,numDeviceEvents_mean,numDeviceEvents_std,numLogonEvents_mean,numLogonEvents_std,numOffHourLogons_mean,numOffHourLogons_std
0,AAE0190,2010-01-04,0.0,2,1,0.0,0.0,2.0,0.0,1.0,0.0
1,AAE0190,2010-01-05,0.0,2,1,0.0,0.0,2.0,0.0,1.0,0.0
2,AAE0190,2010-01-06,0.0,2,1,0.0,0.0,2.0,0.0,1.0,0.0
3,AAE0190,2010-01-07,0.0,2,1,0.0,0.0,2.0,0.0,1.0,0.0
4,AAE0190,2010-01-08,0.0,2,1,0.0,0.0,2.0,0.0,1.0,0.0


In [12]:
# For Device Events
dfZscore["Zscore_numDeviceEvents"] = np.where(
    dfZscore["numDeviceEvents_std"] == 0,
    0,
    (dfZscore["numDeviceEvents"] - dfZscore["numDeviceEvents_mean"]) / dfZscore["numDeviceEvents_std"]
)

# For Logon Events
dfZscore["Zscore_numLogonEvents"] = np.where(
    dfZscore["numLogonEvents_std"] == 0,
    0,
    (dfZscore["numLogonEvents"] - dfZscore["numLogonEvents_mean"]) / dfZscore["numLogonEvents_std"]
)

# For Off-Hour Logons
dfZscore["Zscore_numOffHourLogons"] = np.where(
    dfZscore["numOffHourLogons_std"] == 0,
    0,
    (dfZscore["numOffHourLogons"] - dfZscore["numOffHourLogons_mean"]) / dfZscore["numOffHourLogons_std"]
)
dfZscore = dfZscore[["user","dateOnly","Zscore_numDeviceEvents",	"Zscore_numLogonEvents",	"Zscore_numOffHourLogons"]]
dfZscore.head()

,user,dateOnly,Zscore_numDeviceEvents,Zscore_numLogonEvents,Zscore_numOffHourLogons
0,AAE0190,2010-01-04,0.0,0.0,0.0
1,AAE0190,2010-01-05,0.0,0.0,0.0
2,AAE0190,2010-01-06,0.0,0.0,0.0
3,AAE0190,2010-01-07,0.0,0.0,0.0
4,AAE0190,2010-01-08,0.0,0.0,0.0


In [13]:
dfZscore.shape
mergedAggregated.shape

(330452, 5)

In [14]:
merged = mergedAggregated.merge(dfZscore,on=["user","dateOnly"])
merged.head()

,user,dateOnly,numDeviceEvents,numLogonEvents,numOffHourLogons,Zscore_numDeviceEvents,Zscore_numLogonEvents,Zscore_numOffHourLogons
0,AAE0190,2010-01-04,0.0,2,1,0.0,0.0,0.0
1,AAE0190,2010-01-05,0.0,2,1,0.0,0.0,0.0
2,AAE0190,2010-01-06,0.0,2,1,0.0,0.0,0.0
3,AAE0190,2010-01-07,0.0,2,1,0.0,0.0,0.0
4,AAE0190,2010-01-08,0.0,2,1,0.0,0.0,0.0


In [15]:
#LOADING HTTP DATASET TO ENHANCE THE FEATURE DETECTION
dfHttp = pd.read_csv(os.path.join(path, "r4.2", "http.csv"),usecols=["date","user","url"])
dfHttp.head()

,date,user,url
0,01/02/2010 06:55:16,LRR0148,http://msn.com/The_Human_Centipede_First_Seque...
1,01/02/2010 07:00:13,NGF0157,http://urbanspoon.com/Plunketts_Creek_Loyalsoc...
2,01/02/2010 07:03:46,NGF0157,http://aa.com/Rhodocene/rhodocenium/fhaavatqrf...
3,01/02/2010 07:05:26,IRM0931,http://groupon.com/Leonhard_Euler/leonhard/tne...
4,01/02/2010 07:05:52,IRM0931,http://flickr.com/Inauguration_of_Barack_Obama...


In [16]:
dfHttp["date"] = pd.to_datetime(dfHttp["date"])
dfHttp["dateOnly"] = dfHttp["date"].dt.date
dfHttp["timeOnly"] = dfHttp["date"].dt.time
dfHttp.drop("date",axis=1,inplace=True)

In [17]:
# !ls /kaggle/input/cert-insider-threat-dataset-r4-2/answers/

In [18]:
# !cat /kaggle/input/cert-insider-threat-dataset-r4-2/answers/scenarios.txt

In [19]:
wikileaks_keywords = ['wikileaks',"leaks"]
jobsite_keywords = ['indeed', 'monster', 'careerbuilder', 'linkedin',"job","hire"]
dropbox_keywords = ['dropbox',"storage","cloud","GB","TB"]

dfHttp['isWikileaks'] = dfHttp['url'].str.contains('|'.join(wikileaks_keywords), case=False, na=False)
dfHttp['isJobsite'] = dfHttp['url'].str.contains('|'.join(jobsite_keywords), case=False, na=False)
dfHttp['isDropbox'] = dfHttp['url'].str.contains('|'.join(dropbox_keywords), case=False, na=False)

In [20]:
dfHttp.head()

,user,url,dateOnly,timeOnly,isWikileaks,isJobsite,isDropbox
0,LRR0148,http://msn.com/The_Human_Centipede_First_Seque...,2010-01-02,06:55:16,False,False,False
1,NGF0157,http://urbanspoon.com/Plunketts_Creek_Loyalsoc...,2010-01-02,07:00:13,False,False,True
2,NGF0157,http://aa.com/Rhodocene/rhodocenium/fhaavatqrf...,2010-01-02,07:03:46,False,False,True
3,IRM0931,http://groupon.com/Leonhard_Euler/leonhard/tne...,2010-01-02,07:05:26,False,False,False
4,IRM0931,http://flickr.com/Inauguration_of_Barack_Obama...,2010-01-02,07:05:52,False,False,False


In [21]:
dfHttpAggregated = dfHttp.groupby(["user", "dateOnly"]).agg(
    numWikileaksVisits=("isWikileaks", "sum"),
    numJobsiteVisits=("isJobsite", "sum"),
    numDropboxVisits=("isDropbox", "sum")
)
dfHttpAggregated.reset_index(inplace=True)

In [22]:
merged = merged.merge(dfHttpAggregated,on=["user","dateOnly"],how="outer")
merged.head()

,user,dateOnly,numDeviceEvents,numLogonEvents,numOffHourLogons,Zscore_numDeviceEvents,Zscore_numLogonEvents,Zscore_numOffHourLogons,numWikileaksVisits,numJobsiteVisits,numDropboxVisits
0,AAE0190,2010-01-04,0.0,2,1,0.0,0.0,0.0,0.0,1.0,41.0
1,AAE0190,2010-01-05,0.0,2,1,0.0,0.0,0.0,0.0,3.0,55.0
2,AAE0190,2010-01-06,0.0,2,1,0.0,0.0,0.0,0.0,7.0,66.0
3,AAE0190,2010-01-07,0.0,2,1,0.0,0.0,0.0,0.0,8.0,57.0
4,AAE0190,2010-01-08,0.0,2,1,0.0,0.0,0.0,0.0,2.0,67.0


In [23]:
merged.head()

,user,dateOnly,numDeviceEvents,numLogonEvents,numOffHourLogons,Zscore_numDeviceEvents,Zscore_numLogonEvents,Zscore_numOffHourLogons,numWikileaksVisits,numJobsiteVisits,numDropboxVisits
0,AAE0190,2010-01-04,0.0,2,1,0.0,0.0,0.0,0.0,1.0,41.0
1,AAE0190,2010-01-05,0.0,2,1,0.0,0.0,0.0,0.0,3.0,55.0
2,AAE0190,2010-01-06,0.0,2,1,0.0,0.0,0.0,0.0,7.0,66.0
3,AAE0190,2010-01-07,0.0,2,1,0.0,0.0,0.0,0.0,8.0,57.0
4,AAE0190,2010-01-08,0.0,2,1,0.0,0.0,0.0,0.0,2.0,67.0


In [24]:
#HANDLING THE EMAIL DATASET
dfEmail = pd.read_csv(os.path.join(path, "r4.2", "email.csv"),usecols=["date","user","size","to","attachments"])
dfEmail["date"] = pd.to_datetime(dfEmail["date"])
dfEmail["dateOnly"] = dfEmail["date"].dt.date
dfEmail["timeOnly"] = dfEmail["date"].dt.time
dfEmail.drop("date",axis=1,inplace=True)
dfEmail.head()

,user,to,size,attachments,dateOnly,timeOnly
0,LAP0338,Dean.Flynn.Hines@dtaa.com;Wade_Harrison@lockhe...,25830,0,2010-01-02,07:11:45
1,MOH0273,Odonnell-Gage@bellsouth.net,29942,0,2010-01-02,07:12:16
2,LAP0338,Penelope_Colon@netzero.com,28780,0,2010-01-02,07:13:00
3,LAP0338,Judith_Hayden@comcast.net,21907,0,2010-01-02,07:13:17
4,MOH0273,Bond-Raymond@verizon.net;Alea_Ferrell@msn.com;...,17319,0,2010-01-02,07:13:28


In [25]:
personalMails = ["gmail", "yahoo", "hotmail","outlook"]
dfEmail['isPersonal'] = dfEmail['to'].str.contains('|'.join(personalMails), case=False, na=False)
# Per-user baseline for email size
emailStats = dfEmail.groupby("user").agg(
    mean_size=("size", "mean"),
    stdDev_size=("size", "std")
).reset_index()

dfEmail = dfEmail.merge(emailStats, on="user", how="left")

dfEmail["sizeDeviation"] = np.where(
    dfEmail["stdDev_size"] == 0,
    0,
    (dfEmail["size"] - dfEmail["mean_size"]) / dfEmail["stdDev_size"]
)

# Per-user baseline for attachments
attachStats = dfEmail.groupby("user").agg(
    mean_attach=("attachments", "mean"),
    stdDev_attach=("attachments", "std")
).reset_index()

dfEmail = dfEmail.merge(attachStats, on="user", how="left")

dfEmail["attachDeviation"] = np.where(
    dfEmail["stdDev_attach"] == 0,
    0,
    (dfEmail["attachments"] - dfEmail["mean_attach"]) / dfEmail["stdDev_attach"]
)

dfEmailAggregated = dfEmail.groupby(["user", "dateOnly"]).agg(
    numPersonalMails=("isPersonal", "sum"),
    max_sizeDeviation=("sizeDeviation", "max"),
    mean_sizeDeviation=("sizeDeviation", "mean"),
    max_attachDeviation=("attachDeviation", "max"),
    mean_attachDeviation=("attachDeviation", "mean")
).reset_index()



dfEmailAggregated.head()

,user,dateOnly,numPersonalMails,max_sizeDeviation,mean_sizeDeviation,max_attachDeviation,mean_attachDeviation
0,AAE0190,2010-01-04,0,2.274270,0.150761,2.559498,-0.089923
1,AAE0190,2010-01-05,1,1.206225,-0.267838,0.607293,-0.218640
2,AAE0190,2010-01-06,2,5.082703,0.805028,7.440012,0.467850
3,AAE0190,2010-01-07,1,2.161729,0.389365,2.559498,0.258685
4,AAE0190,2010-01-08,1,2.058214,0.080027,2.559498,0.382038


In [26]:
dfEmail["to"].value_counts(30)

,proportion
to,
Halla.Cathleen.Simmons@dtaa.com,1.165028e-03
Hyatt.Trevor.Hayes@dtaa.com,1.087841e-03
Byron.Tyrone.Williamson@dtaa.com,1.063507e-03
Kirby.Bo.Pollard@dtaa.com,1.032328e-03
Bethany.Xerxes.Johnson@dtaa.com,9.916429e-04
...,...
Amelia.R.Burgess@comcast.net;BJW832@optonline.net,3.802312e-07
Eve.Isadora.Mckenzie@dtaa.com;Stuart.Dennis@hp.com,3.802312e-07
Donovan_G_Nieves@optonline.net;Ramona_Moses@juno.com,3.802312e-07


In [27]:
 merged = merged.merge(dfEmailAggregated,on=["user","dateOnly"],how="outer")
 merged.head()
 merged.fillna(0,inplace=True)

In [28]:
merged.isna().sum()

,0
user,0
dateOnly,0
numDeviceEvents,0
numLogonEvents,0
numOffHourLogons,0
Zscore_numDeviceEvents,0
Zscore_numLogonEvents,0
Zscore_numOffHourLogons,0
numWikileaksVisits,0
numJobsiteVisits,0


In [29]:
dfFile = pd.read_csv(os.path.join(path, "r4.2", "file.csv"),
                     usecols=["date", "user", "pc", "filename"])

dfFile["date"] = pd.to_datetime(dfFile["date"])
dfFile["dateOnly"] = dfFile["date"].dt.date
dfFile["timeOnly"] = dfFile["date"].dt.time

dfFile["ext"] = dfFile["filename"].str.extract(r"\.([^.]+)$")[0].str.lower()

# Step 1: aggregate to user-day level first (raw daily counts)
dfFileAgg = dfFile.groupby(["user", "dateOnly"]).agg(
    numFileEvents=("filename", "count"),
    numUniqueFiles=("filename", "nunique"),
    numZip=("ext", lambda x: (x == "zip").sum()),
    numExe=("ext", lambda x: (x == "exe").sum()),
    numDoc=("ext", lambda x: x.isin(["doc", "docx", "pdf", "txt"]).sum()),
).reset_index()

# Step 2: compute per-user baseline (mean/std) for each raw count column
fileFeatureCols = ["numFileEvents", "numUniqueFiles", "numZip", "numExe", "numDoc"]

userFileStats = dfFileAgg.groupby("user")[fileFeatureCols].agg(["mean", "std"])
userFileStats.columns = ["_".join(col) for col in userFileStats.columns]
userFileStats = userFileStats.reset_index()

# Step 3: bring per-user baseline back onto every user-day row
dfFileAgg = dfFileAgg.merge(userFileStats, on="user", how="left")

# Step 4: compute z-score for each file feature, handling std == 0
for col in fileFeatureCols:
    mean_col = f"{col}_mean"
    std_col = f"{col}_std"
    dfFileAgg[f"Zscore_{col}"] = np.where(
        dfFileAgg[std_col] == 0,
        0,
        (dfFileAgg[col] - dfFileAgg[mean_col]) / dfFileAgg[std_col]
    )

# Step 5: merge only what you need into merged (raw counts + z-scores),
# drop the intermediate mean/std helper columns to avoid clutter
zscoreCols = [f"Zscore_{col}" for col in fileFeatureCols]
dfFileAgg_final = dfFileAgg[["user", "dateOnly"] + fileFeatureCols + zscoreCols]

merged = merged.merge(dfFileAgg_final, on=["user", "dateOnly"], how="outer")
merged.fillna(0, inplace=True)

In [30]:
dfAnswers = pd.read_csv(os.path.join(path, "answers", "insiders.csv"))
# 1. Filter ground truth answers for dataset 4.2
dfAnswers_42 = dfAnswers[dfAnswers["dataset"] == 4.2].copy()

# 2. Convert the start and end timestamps to date objects to match 'dateOnly'
dfAnswers_42["start_date"] = pd.to_datetime(dfAnswers_42["start"]).dt.date
dfAnswers_42["end_date"] = pd.to_datetime(dfAnswers_42["end"]).dt.date

# 3. Initialize the actual_anomaly column as 1 (normal) for all rows
merged["actual_anomaly"] = 1

# 4. Loop through the known malicious events and flag the matching user/date rows as -1
for index, row in dfAnswers_42.iterrows():
    mask = (
        (merged["user"] == row["user"]) &
        (merged["dateOnly"] >= row["start_date"]) &
        (merged["dateOnly"] <= row["end_date"])
    )
    merged.loc[mask, "actual_anomaly"] = -1

In [31]:
merged['actual_anomaly'] = merged['actual_anomaly'].replace(-1, 0)

In [32]:
mergedTemp = merged.copy()

In [33]:
users = merged['user']
merged = merged.drop(['user','dateOnly','numDeviceEvents','numLogonEvents','numOffHourLogons','numFileEvents','numUniqueFiles','numZip','numExe','numDoc'], axis=1)

X = merged.drop('actual_anomaly', axis=1)
y = merged['actual_anomaly']


In [34]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
import pandas as pd


gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)


train_idx, test_idx = next(gss.split(X, y, groups=users))


X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]


scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

In [35]:
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# 1. Compute balanced class weights to handle the imbalanced dataset
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

# 2. Build the ANN model
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2), # Helps prevent overfitting
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid') # Sigmoid for binary classification
])

# 3. Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 4. Train the model
history = model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=512, # A larger batch size is good for your ~264k training samples
    validation_split=0.2,
    class_weight=class_weight_dict
)

Epoch 1/15
415/415 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.7646 - loss: 0.5958 - val_accuracy: 0.8785 - val_loss: 0.5282
Epoch 2/15
415/415 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8444 - loss: 0.5096 - val_accuracy: 0.8613 - val_loss: 0.4790
Epoch 3/15
415/415 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8401 - loss: 0.4831 - val_accuracy: 0.8717 - val_loss: 0.4060
Epoch 4/15
415/415 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8403 - loss: 0.4585 - val_accuracy: 0.8627 - val_loss: 0.4081
Epoch 5/15
415/415 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8386 - loss: 0.4488 - val_accuracy: 0.8642 - val_loss: 0.3825
Epoch 6/15
415/415 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8353 - loss: 0.4460 - val_accuracy: 0.8740 - val_loss: 0.3514
Epoch 7/15
415/415 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8400 - loss: 0.4285 - val_accuracy: 0.8663 - val_loss: 0.3514
Epoch 8/15
415/415 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8375 - loss: 0.4197 - val_accuracy: 0.

In [36]:
# 1. Initialize and train the Random Forest model
# class_weight='balanced' handles your imbalanced dataset automatically
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 2. Get prediction probabilities
# .predict_proba() returns a 2D array. We use [:, 1] to select the probabilities for class 1
rf_pred_prob = rf_model.predict_proba(X_test)[:, 1]



In [39]:
# 3. Apply a custom threshold (using the 0.5 threshold you used for your ANN)
y_pred_rf = (rf_pred_prob > 0.65).astype(int)

# 4. Print the classification report
print(classification_report(y_test, y_pred_rf, target_names=['anomaly (0)', 'normal (1)']))

              precision    recall  f1-score   support

 anomaly (0)       0.93      0.14      0.25       183
  normal (1)       1.00      1.00      1.00     64931

    accuracy                           1.00     65114
   macro avg       0.96      0.57      0.62     65114
weighted avg       1.00      1.00      1.00     65114



In [40]:


# 1. Get prediction probabilities from the ANN
ann_pred_prob = model.predict(X_test)



2035/2035 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step


In [41]:
# 2. Convert probabilities to integer class labels (0 or 1) using a 0.5 threshold
y_pred_ann = (ann_pred_prob > 0.65).astype(int)

# 3. Print the classification report
print(classification_report(y_test, y_pred_ann, target_names=['anomaly (0)', 'normal (1)']))

              precision    recall  f1-score   support

 anomaly (0)       0.01      0.86      0.03       183
  normal (1)       1.00      0.81      0.90     64931

    accuracy                           0.81     65114
   macro avg       0.51      0.84      0.46     65114
weighted avg       1.00      0.81      0.89     65114



In [42]:
from sklearn.metrics import classification_report

# 1. Get prediction probabilities from the ANN
# .flatten() converts the (N, 1) Keras array into a 1D (N,) array to match Random Forest
ann_pred_prob = ann_pred_prob.flatten()

# 2. Average the probabilities from both models
y_pred_prob_avg = (rf_pred_prob + ann_pred_prob) / 2

# 3. Convert probabilities to integer class labels using a 0.5 threshold
# Notice we are now using the new 'y_pred_prob_avg' variable here
y_pred = (y_pred_prob_avg > 0.5).astype(int)

# 4. Print the classification report
print(classification_report(y_test, y_pred, target_names=['anomaly (0)', 'normal (1)']))

              precision    recall  f1-score   support

 anomaly (0)       0.49      0.28      0.36       183
  normal (1)       1.00      1.00      1.00     64931

    accuracy                           1.00     65114
   macro avg       0.74      0.64      0.68     65114
weighted avg       1.00      1.00      1.00     65114

